        # 🧲 L08　正規化與調參
        **統計冒險之旅 2026**　｜　Day 3（09/24 四）🗻 預測之巔　｜　關卡　｜　🏅 100 XP

        📖 ISLP Ch6；資料：勇者咖啡每日營收、會員


        ### 🎯 這一關你會學到
        - Ridge／Lasso：alpha 與係數縮減
- GridSearchCV 用交叉驗證選超參數
- 用 CV 選 max_depth，test 只做一次最終驗收

        ### 🧭 闖關方式
        1. 先按下方「🧰 魔法工具箱」那一格左邊的 ▶（第一次執行 Colab 會花幾秒鐘連線）。
        2. 依序閱讀說明、執行範例、完成每個「🎯 任務」，再執行它下面的「檢查」格。
        3. 看到 ✅ 就往下一個任務；看到 ❌ 就依提示修改，再重新執行任務格與檢查格。
        4. 全部通過後，執行最下面的「🔑 通關密語」格，把密語貼回 [入口網頁](https://johnnychao.github.io/stats-quest-2026/rc/v1.1.0-rc.1/)。

        > 💾 建議先點選「檔案 → 在雲端硬碟中儲存副本」，你的進度才會留在自己的 Google 雲端硬碟。
        > 🎲 這門課的答案常常是小數：任務會告訴你要把答案存進哪個變數，檢查時允許小小的誤差；切分、抽樣、模型請照題目用 `random_state=42`。

In [ ]:
#@title 🧰 魔法工具箱：先在右邊填「暱稱」，再按左邊的 ▶ 執行這一格 { display-mode: "form" }
暱稱 = "" #@param {type:"string"}
# ======================================================================
#  統計冒險之旅 2026 · 關卡檢查工具（看不懂沒關係，這一格不是今天的功課 😉）
# ======================================================================
import hashlib, unicodedata, io, sys, re, contextlib, traceback, builtins, math, warnings
warnings.filterwarnings("ignore")

_LEVEL = "L08"
_COURSE_NAMESPACE = "stats-quest-2026-datama"
_PREFIX = "SQ"
_TASKS = ["8-1", "8-2", "8-3", "8-4", "8-5"]
_XP_EACH = 20
_CHECKS = {}
_PASSED = builtins.__dict__.setdefault("_sq_" + _LEVEL, {})
_HINTS = {}

def _norm_name(s):
    return re.sub(r"\s+", "", unicodedata.normalize("NFKC", str(s))).lower()

def _squash(s):
    return re.sub(r"\s+", "", str(s))

def 出現(out, *subs):
    """輸出中是否（忽略空白）包含所有片段"""
    o = _squash(out)
    return all(_squash(x) in o for x in subs)

def 數字們(out):
    """抓出輸出裡所有的數字（float）"""
    return [float(x) for x in re.findall(r"-?\d+(?:\.\d+)?", str(out))]

# ---------------- 判分器 2.0 ----------------
class _Miss(Exception):
    pass

def 抓變數(ns, name, 型別=None):
    """從任務格執行後的變數取值；沒有就給友善訊息。"""
    if name not in ns:
        raise _Miss(f"我找不到變數 {name}，請確認你有把答案存進名字叫 {name} 的變數（大小寫要一樣）。")
    v = ns[name]
    if 型別 is not None and not isinstance(v, 型別):
        raise _Miss(f"{name} 的型別看起來不對（目前是 {type(v).__name__}）。")
    return v

def _num(v):
    try:
        import numpy as _np
        if hasattr(v, "item"): v = v.item()
    except Exception:
        pass
    return float(v)

def 約等於(v, 目標, 容差=None, 相對=0.01):
    """數值容差：|v-目標| <= 容差（預設為 目標 的 1%，且至少 1e-9）"""
    try:
        x = _num(v)
    except Exception:
        return False
    if x != x:   # NaN
        return False
    tol = 容差 if 容差 is not None else max(abs(目標) * 相對, 1e-9)
    return abs(x - 目標) <= tol

def 資料框像(obj, 列=None, 欄=None, 含欄位=None, 種類="DataFrame"):
    """檢查 DataFrame / Series：列數、欄數、必須包含的欄位；回傳 (ok, 訊息)"""
    import pandas as _pd
    if 種類 == "DataFrame" and not isinstance(obj, _pd.DataFrame):
        return False, f"這應該是一個 DataFrame（目前是 {type(obj).__name__}）。"
    if 種類 == "Series" and not isinstance(obj, _pd.Series):
        return False, f"這應該是一個 Series（目前是 {type(obj).__name__}）。"
    if 列 is not None and len(obj) != 列:
        return False, f"列數應該是 {列}，目前是 {len(obj)}。"
    if 欄 is not None and getattr(obj, "shape", (0, 0))[1] != 欄:
        return False, f"欄數應該是 {欄}，目前是 {obj.shape[1]}。"
    if 含欄位:
        cols = list(obj.columns) if hasattr(obj, "columns") else list(obj.index)
        missing = [c for c in 含欄位 if c not in cols]
        if missing:
            return False, "缺少欄位：" + "、".join(map(str, missing))
    return True, ""


class _NeedMoreInput(Exception):
    pass

_BUILTIN_NAMES = ("sum", "list", "dict", "set", "str", "int", "float", "max", "min", "len",
                  "print", "type", "range", "sorted", "abs", "round", "tuple", "map", "filter",
                  "open", "format", "all", "any", "zip", "bool", "next", "chr", "ord", "id")

_HIST = builtins.__dict__.setdefault("_sq_hist", [])
def _on_pre_run(*args):
    try:
        info = args[0]
        src = getattr(info, "raw_cell", None)
        if isinstance(src, str):
            _HIST.append(src)
    except Exception:
        pass
try:
    _ip = get_ipython()
    if not builtins.__dict__.get("_sq_hooked"):
        _ip.events.register("pre_run_cell", _on_pre_run)
        builtins.__dict__["_sq_hooked"] = True
except Exception:
    pass

def _history():
    try:
        ip = get_ipython()
        h = list(ip.user_ns.get("In") or ip.user_ns.get("_ih") or [])
    except Exception:
        h = list(globals().get("In") or [])
    return [c for c in (h + list(_HIST)) if isinstance(c, str)]

_CALL = re.compile(r"\s*(檢查|通關密語|全部檢查)\s*\(")

def _clean_cell(cell):
    return "\n".join(ln for ln in cell.splitlines() if not _CALL.match(ln))

def _is_mine(cell):
    s = cell.strip()
    if not s:
        return False
    if "#@title" in s or "任務定義(" in s or "_sq_" in s:
        return False
    if _CALL.match(s):
        return False
    return True

def _find_cells(tid):
    marker = "# 🎯 任務 " + tid
    marked = free = None
    im = ifree = -1
    for i, cell in enumerate(_history()):
        if not _is_mine(cell):
            continue
        if marker in cell:
            marked, im = cell, i
        elif "🎯 任務" not in cell:
            free, ifree = cell, i
    return marked, im, free, ifree

def _describe(src):
    body = [ln for ln in src.splitlines() if ln.strip() and not ln.strip().startswith("#")]
    if not body:
        return "（空白）"
    first = body[0].strip()
    return ("%s%s（共 %d 行）" % (first[:52], "…" if len(first) > 52 else "", len(body)))

def _fig_info(_plt):
    out = []
    try:
        for n in _plt.get_fignums():
            f = _plt.figure(n)
            for ax in f.get_axes():
                out.append(dict(title=ax.get_title() or "", xlabel=ax.get_xlabel() or "", ylabel=ax.get_ylabel() or "",
                                n_lines=len(ax.lines), n_patches=len(ax.patches), n_collections=len(ax.collections),
                                legend=bool(ax.get_legend())))
    except Exception:
        pass
    return out

def _make_runner(src):
    def run(*inputs):
        feed = iter([str(x) for x in inputs])
        buf = io.StringIO()
        try:
            ns = dict(get_ipython().user_ns)
        except Exception:
            ns = dict(globals())
        run.shadowed = []
        for _n in _BUILTIN_NAMES:
            _b = getattr(builtins, _n, None)
            if _n in ns and _b is not None and ns[_n] is not _b:
                ns.pop(_n, None)
                run.shadowed.append(_n)
        def _fake_input(prompt=""):
            try:
                return next(feed)
            except StopIteration:
                raise _NeedMoreInput()
        ns["input"] = _fake_input
        ns["__name__"] = "__main__"
        try:
            import matplotlib
            import matplotlib.pyplot as _plt
            _plt.close("all"); _orig_show = _plt.show; _plt.show = lambda *a, **k: None
        except Exception:
            _plt = None
        run.figs = []
        try:
            with contextlib.redirect_stdout(buf):
                exec(compile(src, "<任務 " + _LEVEL + ">", "exec"), ns)
        finally:
            if _plt is not None:
                run.figs = _fig_info(_plt)
                _plt.show = _orig_show
                _plt.close("all")
        return buf.getvalue(), ns
    run.src = src
    run.figs = []
    return run

def 任務定義(tid, fn, 提示=""):
    _CHECKS[tid] = fn
    _HINTS[tid] = 提示

def _fix_shadowed():
    try:
        ns = get_ipython().user_ns
    except Exception:
        ns = globals()
    bad = []
    for n in _BUILTIN_NAMES:
        b = builtins.__dict__.get(n)
        if b is not None and n in ns and ns[n] is not b:
            del ns[n]
            bad.append(n)
    return bad

def _progress():
    done = 0
    total = 0
    for t in _TASKS:
        total += 1
        if _PASSED.get(t):
            done += 1
    bar = "■" * done + "□" * (total - done)
    return f"[{bar}] {done}/{total}"

def _run_check(tid, src):
    run = _make_runner(src)
    try:
        result = _CHECKS[tid](run)
    except _NeedMoreInput:
        return False, "你的程式呼叫 input() 的次數比題目預期的多，請檢查輸入的次數。", []
    except _Miss as e:
        return False, str(e), getattr(run, "shadowed", [])
    except Exception:
        tb = traceback.format_exc().strip().splitlines()[-1]
        return False, "程式執行時發生錯誤 → " + tb, getattr(run, "shadowed", [])
    ok, extra = (result, "") if isinstance(result, bool) else result
    return ok, extra, getattr(run, "shadowed", [])

def _pass(tid):
    first = not _PASSED.get(tid)
    _PASSED[tid] = True
    print(f"✅ 任務 {tid} 通過！{'+' + str(_XP_EACH) + ' XP ' if first else ''}{_progress()}")

def 檢查(tid):
    _shadow = _fix_shadowed()
    tid = builtins.str(tid)
    if tid not in _CHECKS:
        print(f"⚠️ 找不到任務 {tid} 的檢查設定。"); return
    marked, im, free, ifree = _find_cells(tid)
    if marked is None and free is None:
        print(f"❌ 這次執行階段裡，我找不到你寫的程式。")
        print(f"   👉 請先按「# 🎯 任務 {tid}」那一格左邊的 ▶ 執行它，再執行這一格。")
        print("   （如果剛剛重新啟動過執行階段，上面每一格都要重跑一次，包含最上面的魔法工具箱）")
        return
    order = []
    if marked is not None:
        order.append(("標記", marked))
    if free is not None and ifree > im:
        order.append(("最後執行", free))
    if not order:
        order = [("最後執行", free)]
    tried = []
    for kind, src in order:
        ok, extra, shadowed = _run_check(tid, _clean_cell(src))
        tried.append((kind, src, extra, shadowed))
        if ok:
            _pass(tid)
            if extra:
                print("   💬 " + str(extra))
            if _shadow:
                print(f"   ℹ️ 你之前把內建名稱 {'、'.join(_shadow)} 拿來當變數名了，我已經幫你還原。")
                print("      建議換個名字（例如 total、items），不然後面的程式會出現很難懂的錯誤。")
            if kind == "最後執行":
                print(f"   ℹ️ 你的程式最上面少了「# 🎯 任務 {tid}」那一行，我是用你最後執行的那一格判分的。")
                print("      把那一行加回去，之後的檢查會更準確。")
            if shadowed:
                print(f"   ℹ️ 你之前把內建名稱 {'、'.join(shadowed)} 拿來當變數名了，判分時我先幫你還原。")
            if all(_PASSED.get(t) for t in _TASKS):
                print("🏆 本關所有任務都完成了！請執行最下面的「通關密語」那一格。")
            return
    kind, src, extra, shadowed = tried[0]
    print(f"❌ 任務 {tid} 還沒通過。{_progress()}")
    if extra:
        print("   💬 " + str(extra))
    if _HINTS.get(tid):
        print("   💡 提示：" + _HINTS[tid])
    _sh = _shadow + [n for n in shadowed if n not in _shadow]
    if _sh:
        print(f"   ⚠️ 你把內建名稱 {'、'.join(_sh)} 拿來當變數名了（我已還原），這會造成很難懂的錯誤，請改名後重跑那一格。")
    print("   🔎 我判分的是這一段程式：" + _describe(src))
    print(f"      如果這不是你剛剛寫的版本 → 確認第一行的「# 🎯 任務 {tid}」有保留，並重新執行那一格，再按檢查。")

def 全部檢查():
    """出錯或重新啟動執行階段後，重跑完所有任務格，再用這個一次驗收整關。"""
    _fix_shadowed()
    print(f"🔁 重新檢查 {_LEVEL} 的 {len(_TASKS)} 個任務…")
    todo = []
    for t in _TASKS:
        marked, im, free, ifree = _find_cells(t)
        if marked is None and free is None:
            todo.append(t)
            continue
        檢查(t)
    if todo:
        print("⏭️ 這次還沒執行過的任務：" + "、".join(todo))
        print("   先按那幾格左邊的 ▶ 執行，再回來執行 全部檢查()。")

def 通關密語():
    _fix_shadowed()
    missing = [t for t in _TASKS if not _PASSED.get(t)]
    if missing:
        print("🔒 還有任務未通過：" + "、".join(missing) + "　完成後再來拿密語吧！")
        return
    name = 暱稱.strip() if isinstance(暱稱, str) else ""
    if not name:
        name = input("請輸入你在入口網頁登錄的暱稱：").strip()
    if not name:
        print("⚠️ 暱稱不能是空白。"); return
    code = hashlib.sha256(f"{_COURSE_NAMESPACE}|{_LEVEL}|{_norm_name(name)}".encode("utf-8")).hexdigest()[:6].upper()
    print("=" * 46)
    print(f"🎉 恭喜 {name}！{_LEVEL} 通關！")
    print(f"🔑 通關密語：{_PREFIX}-{_LEVEL}-{code}")
    print("👉 回到入口網頁，把密語貼到這一關的「輸入通關密語」欄位。")
    print("=" * 46)

try:
    import numpy as _np_, pandas as _pd_
    _np_.random.seed(42)
except Exception:
    pass
print(f"🧰 魔法工具箱已準備好！本關有 {len(_TASKS)} 個任務：{'、'.join(_TASKS)}")
print("   做完每個任務後，執行它下方的「檢查」格；全部通過後執行最下方的「通關密語」。")

# ---------------- 各任務的檢查規則 ----------------
def _check_8_1(run):
    out, ns = run()
    nz = list(抓變數(ns, "非零數量們"))
    if len(nz) != 6 or nz[0] < nz[-1] or abs(nz[-1] - 1) > 1: return (False, "非零數量 應該隨 alpha 變大而變少，alpha=1000 時只剩 1 個左右。")
    return (sorted(map(str, 抓變數(ns, "倖存者"))) == ['上週同日營收', '假日', '天氣_雨'], "倖存者 = alpha=300 時絕對值最大的 3 個特徵（sorted 後）。")
任務定義("8-1", _check_8_1, 提示="head(3)。")

def _check_8_2(run):
    out, ns = run()
    if not 約等於(抓變數(ns, "最佳alpha"), 0.01, 1e-9): return (False, "最佳alpha = grid.best_params_['ridge__alpha']，且 grid 只能 fit Xd_train。")
    if not 約等於(抓變數(ns, "最佳RMSE"), 1145.530, 2.0): return (False, "最佳RMSE = -grid.best_score_（訓練集內 CV 的負 RMSE）。")
    return (約等於(抓變數(ns, "營收最終測試RMSE"), 951.241, 3.0), "選完 alpha 後，Xd_test 只用這一次最終驗收。")
任務定義("8-2", _check_8_2, 提示="best_score_ 是負數；先用訓練內 CV 選 alpha，最後才評估 Xd_test 一次。")

def _check_8_3(run):
    out, ns = run()
    d = 抓變數(ns, "深度CV_AUC", dict)
    if len(d) != 4: return (False, "深度CV_AUC 要包含 2、5、10、None 四個候選深度。")
    if not 約等於(d.get(5), 0.76401, 0.01): return (False, "深度 5 的 CV AUC 不對：只用 Xm_train、ym_train 做 5-fold CV。")
    return (抓變數(ns, "最佳深度") == 5, "最佳深度要由 CV AUC 最高者決定；Xm_test 依然封存。")
任務定義("8-3", _check_8_3, 提示="用包含類別前處理的 Pipeline 做 cross_val_score；這題不評估 Xm_test。")

def _check_8_4(run):
    out, ns = run()
    p = 抓變數(ns, "最佳參數", dict)
    if set(p.keys()) != {"max_depth", "min_samples_leaf"}: return (False, "最佳參數 = gs.best_params_。")
    return (約等於(抓變數(ns, "最佳CV_AUC"), 0.81120, 0.02), "最佳CV_AUC = gs.best_score_。")
任務定義("8-4", _check_8_4, 提示="best_params_ 與 best_score_。")

def _check_8_5(run):
    out, ns = run()
    if not 約等於(抓變數(ns, "邏輯斯CV_AUC"), 0.82335, 0.01): return (False, "邏輯斯CV_AUC：用 Xm_train, ym_train、cv=5、roc_auc。")
    if str(抓變數(ns, "選擇")) != "邏輯斯": return (False, "差距 < 0.01 → 選簡單的邏輯斯。")
    return (約等於(抓變數(ns, "最終測試AUC"), 0.80745, 0.01), "完成所有 CV 選模後，Xm_test 只評估已選定模型一次。")
任務定義("8-5", _check_8_5, 提示="先完成全部 CV 比較並選模，最後才 fit 整份訓練集、評估 Xm_test 一次。")

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge, Lasso
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, mean_squared_error
daily = pd.read_csv("https://raw.githubusercontent.com/johnnychao/stats-quest-2026/v1.1.0-rc.1/data/coffee_daily.csv")
members = pd.read_csv("https://raw.githubusercontent.com/johnnychao/stats-quest-2026/v1.1.0-rc.1/data/coffee_members.csv")

## 🧲 8-1　正規化：給係數綁彈簧
線索一多（星期 7 欄、分店 3 欄、天氣 3 欄……），最小平方法很貪心，會把每個係數都拉到剛好貼合訓練資料——過度擬合。
**正規化（regularization）**在損失函數裡加一項「係數越大罰越重」，像給每個係數綁一條彈簧，把它們往 0 拉：
- **Ridge**（L2）：所有係數一起縮小，但不會真的變 0。
- **Lasso**（L1）：會把不重要的係數**直接拉到 0** → 順便幫你選特徵。
- **alpha**：彈簧的鬆緊。alpha 越大縮越用力。⚠️ 綁彈簧前一定要標準化，否則單位大的變數吃虧。

本 Notebook 使用 scikit-learn 的 Lasso 慣例：
$\frac{1}{2n}\lVert y-Xw\rVert_2^2+\alpha\lVert w\rVert_1$。
有些教材把 $1/(2n)$ 省略或改用其他係數；那些寫法可透過重新縮放 alpha 得到等價問題，但 **alpha 數值不能跨慣例直接比較**。

> 🧗 模型怎麼找到最好的係數？想像起霧的山上看不到路，摸著腳下的坡度往低處走——這叫梯度下降，想動手刻的去支線 S3。

In [ ]:
#@title 🈶 中文字型設定（畫圖前先執行；Colab 初次約 20～40 秒）
import glob, shutil, subprocess, sys, matplotlib
from matplotlib import font_manager

_font_globs = [
    '/usr/share/fonts/opentype/noto/NotoSansCJK*.ttc',
    '/usr/share/fonts/opentype/noto/NotoSansCJK*.otf',
    'C:/Windows/Fonts/msjh*.ttc',
]
if sys.platform.startswith('linux') and shutil.which('apt-get'):
    if not any(glob.glob(pattern) for pattern in _font_globs[:2]):
        try:
            subprocess.run(
                ['apt-get', '-qq', 'install', '-y', 'fonts-noto-cjk'],
                check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
            )
        except (FileNotFoundError, subprocess.CalledProcessError) as error:
            raise RuntimeError('無法自動安裝中文字型；請確認網路後重新執行本格。') from error
for pattern in _font_globs:
    for path in glob.glob(pattern):
        try:
            font_manager.fontManager.addfont(path)
        except (OSError, RuntimeError):
            pass

_available_fonts = {font.name for font in font_manager.fontManager.ttflist}
_preferred_fonts = [
    'Noto Sans TC', 'Noto Sans CJK TC',
    'Microsoft JhengHei', 'Microsoft JhengHei UI', 'PingFang TC',
    'Noto Sans CJK JP', 'Arial Unicode MS',
]
_chinese_font = next((name for name in _preferred_fonts if name in _available_fonts), None)
if _chinese_font is None:
    raise RuntimeError('找不到可顯示繁體中文的字型；請安裝 Noto Sans CJK 後重新執行本格。')
matplotlib.rcParams['font.family'] = 'sans-serif'
matplotlib.rcParams['font.sans-serif'] = [_chinese_font, 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False
print(f"✅ 中文字型設定完成：{_chinese_font}")

In [ ]:
Xd = daily.drop(columns=["日期", "營收"])
yd = daily["營收"]
Xd_train, Xd_test, yd_train, yd_test = train_test_split(Xd, yd, test_size=0.25, random_state=42)
daily_cat = ["星期", "分店", "天氣"]
daily_num = [c for c in Xd.columns if c not in daily_cat]
daily_preprocess = ColumnTransformer([("cat", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False), daily_cat), ("num", "passthrough", daily_num)], verbose_feature_names_out=False)
daily_scale = Pipeline([("preprocess", daily_preprocess), ("scale", StandardScaler())])
Z = daily_scale.fit_transform(Xd_train)
daily_features = daily_scale.named_steps["preprocess"].get_feature_names_out()
alphas = [0.1, 1, 10, 100, 300, 1000]
路徑 = pd.DataFrame({a: Lasso(alpha=a, max_iter=10000).fit(Z, yd_train).coef_ for a in alphas}, index=daily_features)
print(路徑.round(0))
print("非零係數數量：", [(路徑[a].abs() > 1e-6).sum() for a in alphas])
路徑.T.plot(logx=True, legend=False, title="Lasso：alpha 越大，越多係數被拉到 0"); plt.xlabel("alpha"); plt.show()

## 8-2　alpha 怎麼選？交叉驗證來當裁判
alpha 是**超參數**（模型學不到、要人選的旋鈕）。選法只有一種：把候選值都試一遍，用交叉驗證的分數決定——`GridSearchCV`。

In [ ]:
daily_ridge = Pipeline([("preprocess", daily_preprocess), ("scale", StandardScaler()), ("ridge", Ridge())])
grid = GridSearchCV(daily_ridge, {"ridge__alpha": [0.01, 0.1, 1, 10, 100, 1000]},
                    cv=5, scoring="neg_root_mean_squared_error").fit(Xd_train, yd_train)
print("最佳 alpha：", grid.best_params_, "| 交叉驗證 RMSE：", round(-grid.best_score_, 1))
print(pd.DataFrame(grid.cv_results_)[["param_ridge__alpha", "mean_test_score"]].assign(RMSE=lambda d: -d.mean_test_score).round(1))

## 8-3　樹的深度也是彈簧
決策樹的 `max_depth`、隨機森林的 `min_samples_leaf`，都是「限制模型別太彈性」的旋鈕——和 alpha 是同一件事。深度越深越會背考古題。

流程要守住角色：候選深度、森林調參與邏輯斯比較都只能用**訓練集內的交叉驗證**；所有選擇完成後，鎖住的 **test set 才出場一次**。

In [ ]:
Xm = members.drop(columns=["會員編號", "回購"]); ym = members["回購"]
Xm_train, Xm_test, ym_train, ym_test = train_test_split(Xm, ym, test_size=0.25, random_state=42, stratify=ym)
member_cat = ["性別", "會員等級", "最愛類別"]
member_num = [c for c in Xm.columns if c not in member_cat]
member_preprocess = ColumnTransformer([("cat", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False), member_cat), ("num", "passthrough", member_num)], verbose_feature_names_out=False)
深度CV_AUC範例 = {}
for d in [2, 5, 10, None]:
    候選樹 = Pipeline([("preprocess", member_preprocess), ("model", DecisionTreeClassifier(max_depth=d, random_state=42))])
    深度CV_AUC範例[d] = cross_val_score(候選樹, Xm_train, ym_train, cv=5, scoring="roc_auc").mean()
最佳深度範例 = max(深度CV_AUC範例, key=深度CV_AUC範例.get)
print("訓練集內 CV：", {k: round(v, 3) for k, v in 深度CV_AUC範例.items()}, "→ 選", 最佳深度範例)
print("test 保持鎖住；完成 8-3、8-4、8-5 全部 CV 選擇後才最終驗收。")

## 8-4　簡單優先原則
調完參數的隨機森林如果只比邏輯斯迴歸好一點點（< 0.01），**選簡單的那個**：好解釋、好維護、不容易壞。複雜是有成本的。

模型挑選與簡單優先的判斷都只看相同訓練資料上的 CV；test 不參與選擇。

### 🎯 任務 8-1　Lasso 的係數路徑

對 `alphas = [0.1, 1, 10, 100, 300, 1000]` 各 fit 一個 Lasso（用只由 raw `Xd_train` fit 得的 `Z`），把非零係數數量存成串列 `非零數量們`；alpha=300 時絕對值最大的三個特徵名稱（排序後）存成 `倖存者`。

In [ ]:
# 🎯 任務 8-1　Lasso 的係數路徑（請保留這一行）
alphas = [0.1, 1, 10, 100, 300, 1000]
非零數量們 = []
for a in alphas:
    l = Lasso(alpha=a, max_iter=10000).fit(Z, yd_train)
    非零數量們.append(int((np.abs(l.coef_) > 1e-6).sum()))
l300 = Lasso(alpha=300, max_iter=10000).fit(Z, yd_train)
倖存者 = sorted(pd.Series(l300.coef_, index=daily_features).abs().sort_values(ascending=False).head(???).index)
print(非零數量們, 倖存者)

In [ ]:
檢查("8-1")   # ◀ 執行這一格，看看任務 8-1 有沒有過關

### 🎯 任務 8-2　用交叉驗證選 alpha

對 Ridge 用 `GridSearchCV` 掃 `[0.01, 0.1, 1, 10, 100, 1000]`（Pipeline 含 ColumnTransformer/OneHotEncoder 與 StandardScaler；只對 raw `Xd_train, yd_train` 做 `cv=5`、`scoring='neg_root_mean_squared_error'`），取出 `最佳alpha` 與 `最佳RMSE`。所有選擇完成後，對 `Xd_test` 評估一次並存成 `營收最終測試RMSE`。

In [ ]:
# 🎯 任務 8-2　用交叉驗證選 alpha（請保留這一行）
daily_ridge = Pipeline([("preprocess", daily_preprocess), ("scale", StandardScaler()), ("ridge", Ridge())])
grid = GridSearchCV(daily_ridge, {"ridge__alpha": [0.01, 0.1, 1, 10, 100, 1000]},
                    cv=5, scoring="neg_root_mean_squared_error").fit(Xd_train, yd_train)
最佳alpha = grid.best_params_["ridge__alpha"]
最佳RMSE = ???
營收最終測試RMSE = mean_squared_error(yd_test, grid.best_estimator_.predict(Xd_test)) ** 0.5
print(最佳alpha, round(最佳RMSE, 1), round(營收最終測試RMSE, 1))

In [ ]:
檢查("8-2")   # ◀ 執行這一格，看看任務 8-2 有沒有過關

### 🎯 任務 8-3　用 CV 選樹深

對 `max_depth` 在 `[2, 5, 10, None]` 的每棵候選樹，建立包含 `member_preprocess` 的 Pipeline，只用 raw `Xm_train, ym_train` 做 5-fold ROC AUC 交叉驗證，存成字典 `深度CV_AUC`；以最高 CV AUC 決定 `最佳深度`。`Xm_test` 繼續封存到 8-5。

In [ ]:
# 🎯 任務 8-3　樹的深度（請保留這一行）
深度CV_AUC = {}
for d in [2, 5, 10, None]:
    候選樹 = Pipeline([("preprocess", member_preprocess), ("model", DecisionTreeClassifier(max_depth=d, random_state=42))])
    深度CV_AUC[d] = cross_val_score(候選樹, Xm_train, ym_train, cv=5, scoring=???).mean()
最佳深度 = max(深度CV_AUC, key=深度CV_AUC.get)
print({k: round(v, 3) for k, v in 深度CV_AUC.items()}, 最佳深度, "test 仍封存")

In [ ]:
檢查("8-3")   # ◀ 執行這一格，看看任務 8-3 有沒有過關

### 🎯 任務 8-4　GridSearchCV 調隨機森林

用包含 `member_preprocess` 與 `RandomForestClassifier(200, random_state=42)` 的 Pipeline 做 `GridSearchCV`，掃 `{'max_depth': [3, 5, 8, None], 'min_samples_leaf': [1, 5, 20]}`（`cv=5`、`scoring='roc_auc'`，只用 raw 訓練集 fit），取出去掉 `model__` 前綴的 `最佳參數`（字典）與 `最佳CV_AUC`。

In [ ]:
# 🎯 任務 8-4　GridSearchCV 調隨機森林（請保留這一行）
rf_pipe = Pipeline([("preprocess", member_preprocess), ("model", RandomForestClassifier(200, random_state=42))])
gs = GridSearchCV(rf_pipe,
                  {"model__max_depth": [3, 5, 8, None], "model__min_samples_leaf": [1, 5, 20]},
                  cv=5, scoring="roc_auc").fit(Xm_train, ym_train)
最佳參數 = {k.replace("model__", ""): v for k, v in ???.items()}
最佳CV_AUC = ???
print(最佳參數, round(最佳CV_AUC, 3))

In [ ]:
檢查("8-4")   # ◀ 執行這一格，看看任務 8-4 有沒有過關

### 🎯 任務 8-5　簡單優先

算出包含 `member_preprocess` 的邏輯斯迴歸 Pipeline 在**raw 訓練集**上的 5 摺 AUC 平均 `邏輯斯CV_AUC`，與 `最佳CV_AUC` 相減得 `差距`；若差距 < 0.01 就把 `選擇` 設成 `"邏輯斯"`，否則 `"隨機森林"`。完成這個最後選擇後，將 `最終會員模型` fit 全部訓練集，只用 `Xm_test` 評估一次 `最終測試AUC`。

In [ ]:
# 🎯 任務 8-5　簡單優先（請保留這一行）
pipe = Pipeline([("preprocess", member_preprocess), ("scale", StandardScaler()), ("model", LogisticRegression(max_iter=1000))])
邏輯斯CV_AUC = cross_val_score(pipe, Xm_train, ym_train, cv=5, scoring="roc_auc").mean()
差距 = 最佳CV_AUC - 邏輯斯CV_AUC
選擇 = ???
最終會員模型 = pipe if 選擇 == "邏輯斯" else gs.best_estimator_
最終會員模型.fit(Xm_train, ym_train)
最終測試AUC = roc_auc_score(ym_test, 最終會員模型.predict_proba(Xm_test)[:, 1])
print(round(邏輯斯CV_AUC, 3), round(差距, 3), 選擇, round(最終測試AUC, 3))

In [ ]:
檢查("8-5")   # ◀ 執行這一格，看看任務 8-5 有沒有過關

## 🌟 進階挑戰（不計分）
1. 把 8-2 的 Ridge 換成 Lasso，最佳 alpha 是多少？留下哪些特徵？
2. 在 8-4 加入 `n_estimators: [50, 200, 500]`，分數有差嗎？花了多少時間？

---
## 🔑 通關密語
　你已經會用交叉驗證幫模型調旋鈕，也知道什麼時候該選簡單的。
全部任務都 ✅ 之後，執行下面這一格，會得到你專屬的通關密語（和暱稱綁定，每個人不一樣）。

In [ ]:
通關密語()

---
### 🧭 接下來
**下一關：🧭 L09 沒有標準答案的學習：PCA 與 K-means** → [在 Colab 開啟](https://colab.research.google.com/github/johnnychao/stats-quest-2026/blob/v1.1.0-rc.1/notebooks/L09_pca_kmeans.ipynb)

回到入口網頁：https://johnnychao.github.io/stats-quest-2026/rc/v1.1.0-rc.1/